# 01 · Fetch Data

抓兩種資料：
1. 台南市村里界圖（GeoJSON / SHP）
2. 台南市立圖書館清單與經緯度

成功後輸出：
- `data/raw/tainan_villages.geojson`
- `data/raw/tainan_libraries.csv`

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests

# 讓 notebook 找得到 lib/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
FALLBACK_DIR = ROOT / "data" / "fallback"
RAW_DIR.mkdir(parents=True, exist_ok=True)

VILLAGES_OUT = RAW_DIR / "tainan_villages.geojson"
LIBRARIES_OUT = RAW_DIR / "tainan_libraries.csv"

print(f"ROOT = {ROOT}")
print(f"RAW_DIR = {RAW_DIR}")

## 1. 村里界圖

來源：政府資料開放平台「村里界圖」（內政部國土測繪中心）

由於該資料集 URL 經常變動且檔案較大（~50MB），預設**手動下載**：

1. 開 https://data.gov.tw/ 搜尋「村里界圖」（或上 https://segis.moi.gov.tw/ ）
2. 下載 WGS84 經緯度版本（SHP 或 GeoJSON 皆可）
3. 解壓後將檔案放到 `data/raw/`，下一個 cell 會自動讀取並過濾台南市

In [ ]:
# 自動偵測 data/raw/ 內任何 .shp 或 .geojson（除了我們自己輸出的）
candidates = [
    p for p in RAW_DIR.glob("*")
    if p.suffix.lower() in (".shp", ".geojson", ".gpkg")
    and p.name != VILLAGES_OUT.name
]

if not candidates:
    raise FileNotFoundError(
        f"找不到村里界圖。請依上一個 cell 說明手動下載到 {RAW_DIR}"
    )

src = candidates[0]
print(f"讀取 {src.name}...")
gdf = gpd.read_file(src)
print(f"全國共 {len(gdf)} 個里")
print(f"欄位：{list(gdf.columns)}")
gdf.head(2)

In [ ]:
# 內政部資料的欄位名稱可能是 COUNTYCODE / COUNTY_ID / COUNTY 之一；遇到都嘗試
county_col_candidates = ["COUNTYCODE", "COUNTY_ID", "COUNTY"]
county_col = next((c for c in county_col_candidates if c in gdf.columns), None)
if county_col is None:
    raise KeyError(
        f"找不到縣市代碼欄位（試過 {county_col_candidates}）；"
        f"現有欄位：{list(gdf.columns)}"
    )

# 台南：代碼以 67 開頭或名稱含「臺南」/「台南」
if gdf[county_col].dtype == object:
    mask = gdf[county_col].astype(str).str.contains("臺南|台南", na=False) | \
           gdf[county_col].astype(str).str.startswith("67")
else:
    mask = gdf[county_col].astype(str).str.startswith("67")

tainan = gdf[mask].copy()
print(f"台南市共 {len(tainan)} 個里")

# 確保 CRS 是 WGS84
if tainan.crs is None or tainan.crs.to_epsg() != 4326:
    tainan = tainan.to_crs(epsg=4326)

# 給每個里一個穩定的 village_id；優先使用 VILLCODE
id_col = next(
    (c for c in ["VILLCODE", "VILLAGE_ID", "VILLAGE_CODE"] if c in tainan.columns),
    None,
)
if id_col is None:
    tainan["village_id"] = [f"tn_{i:04d}" for i in range(len(tainan))]
else:
    tainan["village_id"] = tainan[id_col].astype(str)

# 統一里名 / 區名欄位
name_col = next((c for c in ["VILLNAME", "VILLAGE", "NAME"] if c in tainan.columns), None)
district_col = next((c for c in ["TOWNNAME", "TOWN"] if c in tainan.columns), None)

tainan["village_name"] = tainan[name_col] if name_col else ""
tainan["district"] = tainan[district_col] if district_col else ""

# 只保留必要欄位 + 幾何
out = tainan[["village_id", "village_name", "district", "geometry"]]
out.to_file(VILLAGES_OUT, driver="GeoJSON")
print(f"✅ Saved {len(out)} villages to {VILLAGES_OUT}")

## 2. 圖書館清單

策略：依序嘗試
1. 臺南市政府開放資料平台 https://data.tainan.gov.tw/ (`圖書館各館資訊`)
2. 內建備援 `data/fallback/libraries_hardcoded.json`

備援清單已涵蓋 37 區的主要館舍，不需爬蟲也能跑完整個流程。

In [ ]:
def try_tainan_open_data() -> pd.DataFrame | None:
    """Try Tainan open data API; return DataFrame or None on failure."""
    # 此處為示意；實際 API endpoint 可能變動，找不到就 return None
    # 主要 dataset 名稱常見：圖書館分館資訊 / 臺南市立圖書館各分館
    try:
        url = "https://data.tainan.gov.tw/api/3/action/datastore_search"
        # 占位：實際使用前請去 data.tainan.gov.tw 搜尋「圖書館」確認 resource_id
        # 找到後填入下面這個變數，否則直接返回 None
        resource_id = ""
        if not resource_id:
            return None
        resp = requests.get(url, params={"resource_id": resource_id, "limit": 200}, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        records = data.get("result", {}).get("records", [])
        if not records:
            return None
        # 假設欄位有 name / address / lat / lon —— 視實際資料調整
        df = pd.DataFrame(records)
        return df
    except Exception as e:
        print(f"⚠️  Tainan open data fetch failed: {e}")
        return None


def load_fallback() -> pd.DataFrame:
    with (FALLBACK_DIR / "libraries_hardcoded.json").open(encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data["libraries"])


libs = try_tainan_open_data()
if libs is None:
    print("ℹ️  Using hardcoded fallback library list")
    libs = load_fallback()
else:
    print(f"✅ Fetched {len(libs)} libraries from Tainan open data")

# 確保 schema 一致
required = {"name", "district", "address", "lat", "lon"}
missing = required - set(libs.columns)
if missing:
    raise ValueError(f"Library data missing columns: {missing}")

libs = libs[["name", "district", "address", "lat", "lon"]].copy()
libs.to_csv(LIBRARIES_OUT, index=False, encoding="utf-8-sig")  # utf-8-sig 讓 Excel 看中文
print(f"✅ Saved {len(libs)} libraries to {LIBRARIES_OUT}")
libs.head()

In [ ]:
import matplotlib.pyplot as plt

villages = gpd.read_file(VILLAGES_OUT)
libs_gdf = gpd.GeoDataFrame(
    libs,
    geometry=gpd.points_from_xy(libs.lon, libs.lat),
    crs="EPSG:4326",
)

fig, ax = plt.subplots(figsize=(8, 9))
villages.boundary.plot(ax=ax, linewidth=0.2, color="gray")
libs_gdf.plot(ax=ax, color="red", markersize=20)
ax.set_title(f"Tainan: {len(villages)} villages + {len(libs_gdf)} libraries")
ax.set_aspect("equal")
plt.show()